In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 5 - WEEK 9 BAYESIAN OPTIMISATION
# Run from inside the week9/ folder
# ============================================================

# ------------------------------------------------------------
# 1. Load cumulative Week 9 data
# ------------------------------------------------------------

X = np.load("function5/initial_inputs.npy")
Y = np.load("function5/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 8 calibration check
# ------------------------------------------------------------
#
# Week 8 selected:
# [1, 1, 1, 0.9899924]
#
# GP prediction:
# mean ≈ 8237.870
# std  ≈ 343.886
#
# Actual:
# 8472.483647641197
# ------------------------------------------------------------

week8_pred_mean = 8237.87021279142
week8_pred_std = 343.88592351500284
week8_actual = 8472.483647641197

week8_error = week8_actual - week8_pred_mean
week8_z_error = week8_error / week8_pred_std

print("\n================================")
print("WEEK 8 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week8_pred_mean)
print("Predicted std :", week8_pred_std)
print("Actual        :", week8_actual)

print("\nPrediction error:")
print(week8_error)

print("\nError / predicted std:")
print(week8_z_error)


# ------------------------------------------------------------
# 3. Standardise Y manually
# ------------------------------------------------------------

y_mean = np.mean(Y)
y_std = np.std(Y)

Y_scaled = (Y - y_mean) / y_std
best_y_scaled = np.max(Y_scaled)

print("\nY mean:", y_mean)
print("Y std :", y_std)


# ------------------------------------------------------------
# 4. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y_scaled)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Candidate generation
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal widths:", local_scale)
print("Wide widths:", wide_scale)

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(90000, 4)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(60000, 4)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(100000, 4)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

# Remove near-duplicates

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 7. GP predictions
# ------------------------------------------------------------

mu_scaled, sigma_scaled = gp.predict(
    candidates,
    return_std=True
)

mu_raw = (
    mu_scaled * y_std
    + y_mean
)

sigma_raw = (
    sigma_scaled * y_std
)


# ------------------------------------------------------------
# 8. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu_scaled,
    sigma_scaled,
    best_y_scaled,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("raw mean =", mu_raw[ei_idx])
print("raw std =", sigma_raw[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 9. EI sensitivity
# ------------------------------------------------------------

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in [0.0, 0.01, 0.05, 0.10]:

    EI_test = expected_improvement(
        mu_scaled,
        sigma_scaled,
        best_y_scaled,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", xi,
        "\n candidate =", candidates[idx],
        "\n raw mean =", round(mu_raw[idx], 3),
        "\n raw std =", round(sigma_raw[idx], 3),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 10. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu_scaled)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("raw mean =", mu_raw[mean_idx])
print("raw std =", sigma_raw[mean_idx])


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = mu_scaled + beta * sigma_scaled
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n raw mean =", round(mu_raw[idx], 3),
        "\n raw std =", round(sigma_raw[idx], 3),
        "\n scaled UCB =", round(UCB[idx], 6),
        "\n"
    )

X shape: (28, 4)
Y shape: (28,)

Current best:
[1. 1. 1. 1.] -> 8662.4825

Y range:
min = 0.1129397953712203
max = 8662.4825
std = 2732.0339040407625

WEEK 8 CALIBRATION CHECK
Predicted mean: 8237.87021279142
Predicted std : 343.88592351500284
Actual        : 8472.483647641197

Prediction error:
234.61343484977624

Error / predicted std:
0.6822420425113466

Y mean: 1671.614373487572
Y std : 2732.0339040407625


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
1.81**2 * Matern(length_scale=[1.15, 1.2, 1.15, 2], nu=2.5) + WhiteKernel(noise_level=0.00169)

ARD lengthscales:
[1.14608003 1.19599426 1.14620125 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.28318987 0.27137108 0.28315992 0.16227913]

Local widths: [0.1 0.1 0.1 0.1]
Wide widths: [0.2 0.2 0.2 0.2]

Candidates after duplicate filtering:
235768

PRIMARY EI
candidate = [1.         1.         1.         0.25164852]
raw mean = 6036.596044064705
raw std = 1322.6085116189593
EI = 0.004274233247547724

EI SENSITIVITY

xi = 0.0 
 candidate = [1.         1.         1.         0.25164852] 
 raw mean = 6036.596 
 raw std = 1322.609 
 EI = 0.00427423 

xi = 0.01 
 candidate = [1.         1.         1.         0.25164852] 
 raw mean = 6036.596 
 raw std = 1322.609 
 EI = 0.00404439 

xi = 0.05 
 candidate = [1.         1.         1.         0.25164852] 
 raw mean = 6036.596 
 raw std = 1322.609 
 EI = 0.00323078 

xi = 0.1 
 candidate = [1.         1.         

In [2]:
# ============================================================
# FINAL FUNCTION 5 - WEEK 9 SELECTION
# ============================================================
#
# Week 8 calibration error was only +0.68 predictive standard
# deviations, supporting trust in the local GP fit.
#
# EI is rejected because it moves far down x4 with a much
# lower predicted mean and very high uncertainty.
#
# Highest mean and every tested UCB beta select exactly the
# same near-boundary candidate.
#
# This region is also supported by two strong realised
# observations from Weeks 7 and 8.

mean_idx = np.argmax(mu_scaled)

week9_candidate = candidates[mean_idx]

print("Week 9 Function 5 candidate:")
print(week9_candidate)

print("\nPredicted raw mean:")
print(mu_raw[mean_idx])

print("\nPredicted raw std:")
print(sigma_raw[mean_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week9_candidate
)

print("\nPortal format:")
print(portal)

Week 9 Function 5 candidate:
[1.         1.         1.         0.97999072]

Predicted raw mean:
8460.837479980311

Predicted raw std:
137.98176799506345

Portal format:
1.000000-1.000000-1.000000-0.979991
